# Week 38: Language Modelling

In [1]:
import nb_setup

from src.data import load_tokenised_data
from src.lm_utils import build_vocab, apply_unk, UnigramLM, BigramLM, TrigramLM

import pandas as pd
import math

LANG = "ar"

df_train_tokenised, df_val_tokenised = load_tokenised_data()

train_df = df_train_tokenised.loc[df_train_tokenised["lang"] == LANG]
val_df   = df_val_tokenised.loc[df_val_tokenised["lang"] == LANG]

ar_train = train_df["question_tokens"].tolist()
ar_val   = val_df["question_tokens"].tolist()

In [2]:
vocab = build_vocab(ar_train, min_count=2)

ar_train_unk = [apply_unk(toks, vocab) for toks in ar_train]
ar_val_unk   = [apply_unk(toks, vocab) for toks in ar_val]

print("Vocab size (incl. <UNK>/<s>/</s>):", len(vocab))

Vocab size (incl. <UNK>/<s>/</s>): 1723


In [3]:
def unk_rate(token_lists):
    total = sum(len(t) for t in token_lists)
    unks = sum(t.count("<UNK>") for t in token_lists)
    return unks / total

print("Val UNK rate:", unk_rate(ar_val_unk))

Val UNK rate: 0.2798029556650246


**Unigram Modelling with Add-k Smoothing**

In [4]:
for k in [0.001, 0.005, 0.01, 0.02, 0.05]:
    m = UnigramLM(k=k)
    m.fit(ar_train_unk, vocab)
    print(f"k={k}: val perplexity = {m.perplexity(ar_val_unk):.2f}")

k=0.001: val perplexity = 80.81
k=0.005: val perplexity = 80.82
k=0.01: val perplexity = 80.83
k=0.02: val perplexity = 80.85
k=0.05: val perplexity = 80.92


In [5]:
BEST_K = 0.001 # lowest value
model = UnigramLM(k=BEST_K)
model.fit(ar_train_unk, vocab)
ppl = model.perplexity(ar_val_unk)
print(f"Unigram (k={BEST_K}) validation perplexity:", ppl)

Unigram (k=0.001) validation perplexity: 80.81417687491928


In [6]:
def question_perplexity(model, tokens):
    logprob = model.sentence_logprob(tokens)
    return math.exp(-logprob / len(tokens))

scored = sorted(
    zip(ar_val, ar_val_unk),
    key=lambda pair: question_perplexity(model, pair[1])
)
print("Lowest perplexity (most predictable):", scored[:3])
print("Highest perplexity (most surprising):", scored[-3:])

Lowest perplexity (most predictable): [(array(['متى', 'بنيت', 'الكنيسة', 'القبطية', 'الأرثوذكسية', 'المشرقية',
       '؟'], dtype=object), ['متى', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '؟']), (array(['متى', 'أطيح', 'بالرئيس', 'السوداني', 'البشير', '؟'], dtype=object), ['متى', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '؟']), (array(['متى', 'شهد', 'ظهور', 'أنونيموس', '؟'], dtype=object), ['متى', '<UNK>', '<UNK>', '<UNK>', '؟'])]
Highest perplexity (most surprising): [(array(['متى', 'وصل', 'الاسلام', 'إلى', 'قارة', 'أمريكا', 'الشمالية', '؟'],
      dtype=object), ['متى', 'وصل', 'الاسلام', 'إلى', 'قارة', 'أمريكا', 'الشمالية', '؟']), (array(['من', 'إكتشف', 'أمريكا'], dtype=object), ['من', 'إكتشف', 'أمريكا']), (array(['من', 'إكتشف', 'أمريكا'], dtype=object), ['من', 'إكتشف', 'أمريكا'])]


In [7]:
def question_unk_rate(unk_tokens):
    return unk_tokens.count("<UNK>") / len(unk_tokens)

rows = []
for orig, unk in zip(ar_val, ar_val_unk):
    rows.append({
        "unk_rate": question_unk_rate(unk),
        "perplexity": question_perplexity(model, unk),
        "length": len(unk),
    })

diag_df = pd.DataFrame(rows)
print(diag_df[["unk_rate", "perplexity"]].corr())

            unk_rate  perplexity
unk_rate     1.00000    -0.68193
perplexity  -0.68193     1.00000


In [8]:
# perplexity on full validation set
full_ppl = model.perplexity(ar_val_unk)

# perplexity on de-duplicated validation set
dedup_mask = ~val_df["question"].duplicated()
ar_val_dedup_unk = [unk for unk, keep in zip(ar_val_unk, dedup_mask) if keep]
dedup_ppl = model.perplexity(ar_val_dedup_unk)

print("Full val perplexity:", full_ppl)
print("De-duplicated val perplexity:", dedup_ppl)

Full val perplexity: 80.81417687491928
De-duplicated val perplexity: 79.47755631991346


In [9]:
results_row_amna = {
    "member": "amna",
    "language": LANG,
    "model": "unigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}
results_row_amna

{'member': 'amna',
 'language': 'ar',
 'model': 'unigram',
 'smoothing': 'add-k (k=0.001)',
 'vocab_size': 1723,
 'val_unk_rate': 0.2798029556650246,
 'val_perplexity_full': 80.81417687491928,
 'val_perplexity_dedup': 79.47755631991346}

# Results of Unigram with Add-K Smoothing

We implement a unigram baseline with add-k smoothing over Arabic questions, using a shared vocabulary built from training tokens (min-count 2, folding rarer tokens into <UNK>; vocab size 1,723). Sweeping k ∈ {0.001, 0.005, 0.01, 0.02, 0.05} shows validation perplexity increasing monotonically with k (80.81 at k=0.001 to 86.12 at k=2.0), with diminishing returns below k=0.01; we select k=0.001. We attribute this to the vocabulary/UNK strategy already absorbing most out-of-vocabulary risk (28.0% of validation tokens map to <UNK>), leaving little for additional smoothing to protect against, so heavier smoothing chiefly reallocates probability mass away from well-attested words. Per-question analysis shows a strong negative correlation (r = −0.68) between a question's <UNK> rate and its perplexity: questions dominated by <UNK> register as "predictable" because <UNK> itself is a high-frequency token in the model, not because the underlying question is linguistically simple a limitation of unigram modelling worth noting, since it rewards lexical genericness over topical specificity. The validation set contains 74/415 (17.8%) duplicate questions; de-duplicating changes perplexity only modestly (80.81 → 79.48), indicating the headline number is not substantially inflated by repeated examples.

### Bigram model with Add-K Smoothing

In [10]:
bi_gram_ks = [0.001, 0.005, 0.01, 0.02, 0.05]
ppl_per_k = {}
for k in bi_gram_ks:
    model = BigramLM(k=k)

    model.fit(ar_train_unk, vocab)

    ppl = model.perplexity(ar_val_unk)
    ppl_per_k[k] = ppl

    print(
        f"k={k}: "
        f"validation perplexity = {ppl:.2f}"
    )

k=0.001: validation perplexity = 35.76
k=0.005: validation perplexity = 34.28
k=0.01: validation perplexity = 35.73
k=0.02: validation perplexity = 38.68
k=0.05: validation perplexity = 45.44


In [11]:
BEST_K = min(ppl_per_k, key=ppl_per_k.get)

model = BigramLM(k=BEST_K)

model.fit(ar_train_unk, vocab)

full_ppl = model.perplexity(ar_val_unk)

In [12]:
dedup_mask = ~val_df["question"].duplicated()

ar_val_dedup_unk = [
    unk
    for unk, keep in zip(ar_val_unk, dedup_mask)
    if keep
]

dedup_ppl = model.perplexity(
    ar_val_dedup_unk
)

print("Bigram Full validation perplexity:", full_ppl)
print("De-duplicated validation perplexity:", dedup_ppl)

Bigram Full validation perplexity: 34.283527528389705
De-duplicated validation perplexity: 33.480258985434695


In [13]:
scored = [
    (
        original,
        unk,
        question_perplexity(model, unk),
    )
    for original, unk
    in zip(ar_val, ar_val_unk)
]

scored = sorted(scored, key=lambda x: x[2])

In [14]:
for original, unk, ppl in scored[:5]:
    print(f"PPL={ppl:.2f}", original)

PPL=3.98 ['ما' 'هي' 'الرقصة' 'الشركسية' '؟']
PPL=3.98 ['ما' 'هي' 'الفلسفة' 'التشاؤمية' '؟']
PPL=4.17 ['هل' 'تنتج' 'شجرة' 'السماء' 'ثماراً' '؟']
PPL=4.25 ['متى' 'بنيت' 'الكنيسة' 'القبطية' 'الأرثوذكسية' 'المشرقية' '؟']
PPL=4.25 ['من' 'هو' 'كلاشنكوف' '؟']


In [15]:
for original, unk, ppl in scored[-5:]:
    print(f"PPL={ppl:.2f}", original)

PPL=511.98 ['من' 'تولى' 'رئاسة' 'العراق' 'بعد' 'الاستقلال' '؟']
PPL=511.98 ['من' 'تولى' 'رئاسة' 'العراق' 'بعد' 'الاستقلال' '؟']
PPL=559.60 ['من' 'وعد' 'اليهود' 'بأرض' 'فلسطين' '؟']
PPL=559.60 ['من' 'وعد' 'اليهود' 'بأرض' 'فلسطين' '؟']
PPL=803.17 ['كم' 'سنة' 'ميلادية' 'عاش' 'السيد' 'المسيح' '؟']


In [16]:
results_row_willy = {
    "member": "willy",
    "language": LANG,
    "model": "bigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}

results_row_willy

{'member': 'willy',
 'language': 'ar',
 'model': 'bigram',
 'smoothing': 'add-k (k=0.005)',
 'vocab_size': 1723,
 'val_unk_rate': 0.2798029556650246,
 'val_perplexity_full': 34.283527528389705,
 'val_perplexity_dedup': 33.480258985434695}

### Trigram Model with Add-K Smoothing

In [17]:
trigram_ks = [0.001, 0.005, 0.01, 0.02, 0.05]

ppl_per_k = {}

for k in trigram_ks:
    model = TrigramLM(k=k)
    model.fit(ar_train_unk, vocab)
    ppl = model.perplexity(ar_val_unk)

    ppl_per_k[k] = ppl
    print(f"k={k}: validation perplexity = {ppl:.2f}")

k=0.001: validation perplexity = 77.79
k=0.005: validation perplexity = 76.21
k=0.01: validation perplexity = 80.54
k=0.02: validation perplexity = 88.28
k=0.05: validation perplexity = 105.14


In [18]:
BEST_K = min(ppl_per_k, key=ppl_per_k.get)
print("Best k:", BEST_K)

Best k: 0.005


In [19]:
model = TrigramLM(k=BEST_K)
model.fit(ar_train_unk, vocab)

In [20]:
full_ppl = model.perplexity(ar_val_unk)

In [21]:
dedup_mask = ~val_df["question"].duplicated()

ar_val_dedup_unk = [
    unk
    for unk, keep in zip(ar_val_unk, dedup_mask)
    if keep
]

dedup_ppl = model.perplexity(ar_val_dedup_unk)

print("Trigram Full validation perplexity:", full_ppl)
print("De-duplicated validation perplexity:", dedup_ppl)

Trigram Full validation perplexity: 76.21452459002472
De-duplicated validation perplexity: 75.17549271335143


In [22]:
scored = [
    (
        original,
        unk,
        question_perplexity(model, unk),
    )
    for original, unk
    in zip(ar_val, ar_val_unk)
]

scored = sorted(scored, key=lambda x: x[2])

In [23]:
# 5 most predictable questions
for original, unk, ppl in scored[:5]:
    print(f"PPL={ppl:.2f}", original)

PPL=3.69 ['ما' 'هو' 'الصاروخ' 'الشيطان' '؟']
PPL=3.69 ['ما' 'هو' 'العظم' 'السنخي' '؟']
PPL=3.69 ['ما' 'هو' 'الصاروخ' 'الشيطان' '؟']
PPL=3.72 ['ما' 'هي' 'الرقصة' 'الشركسية' '؟']
PPL=3.72 ['ما' 'هي' 'الفلسفة' 'التشاؤمية' '؟']


In [24]:
# 5 most surprising questions
for original, unk, ppl in scored[-5:]:
    print(f"PPL={ppl:.2f}", original)

PPL=1388.16 ['كم' 'سنة' 'ميلادية' 'عاش' 'السيد' 'المسيح' '؟']
PPL=1451.22 ['هل' 'تقنية' 'الجيل' 'الخامس' 'تعتبر' 'خطرة' '؟']
PPL=1471.51 ['ماهي' 'آخر' 'الكولونيات' 'الفرنسية' 'في' 'أفريقيا' '؟']
PPL=2532.64 ['كم' 'فازت' 'الصين' 'بميدالية' 'بدورة' 'الألعاب' 'الأولمبية' '2012' '؟']
PPL=2622.40 ['ماهو' 'معنى' 'بروتكول' 'الإتش' 'تي' 'تي' 'بي' '؟']


In [25]:
results_row_zarin = {
    "member": "zarin",
    "language": LANG,
    "model": "trigram",
    "smoothing": f"add-k (k={BEST_K})",
    "vocab_size": len(vocab),
    "val_unk_rate": unk_rate(ar_val_unk),
    "val_perplexity_full": full_ppl,
    "val_perplexity_dedup": dedup_ppl,
}

results_row_zarin

{'member': 'zarin',
 'language': 'ar',
 'model': 'trigram',
 'smoothing': 'add-k (k=0.005)',
 'vocab_size': 1723,
 'val_unk_rate': 0.2798029556650246,
 'val_perplexity_full': 76.21452459002472,
 'val_perplexity_dedup': 75.17549271335143}

In [26]:
results_df = pd.DataFrame([
    results_row_amna,
    results_row_willy,
    results_row_zarin
])
results_df[
    ["val_unk_rate", "val_perplexity_full", "val_perplexity_dedup"]
] = results_df[
    ["val_unk_rate", "val_perplexity_full", "val_perplexity_dedup"]
].round(4)

results_df

,member,language,model,smoothing,vocab_size,val_unk_rate,val_perplexity_full,val_perplexity_dedup
0,amna,ar,unigram,add-k (k=0.001),1723,0.2798,80.8142,79.4776
1,willy,ar,bigram,add-k (k=0.005),1723,0.2798,34.2835,33.4803
2,zarin,ar,trigram,add-k (k=0.005),1723,0.2798,76.2145,75.1755
